# Customer churn occurs when a subscriber stops using a company's service during a given period, resulting in the loss of recurring revenue.

In [3]:
import pandas as pd

df = pd.read_csv("C:/Users/HP/Desktop/dataanlytics/churn analysis/data/churn_raw.csv")

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
df.shape


(7043, 21)

In [5]:

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [6]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


# Project Planning & Initial Inspection

## 1. Dataset Overview

The **Telco Customer Churn** dataset provides a snapshot of a fictional telecommunications company that provided home phone and Internet services to 7,043 customers in California.

* **Total Rows:** 7,043
* **Total Columns:** 21
* **Target Variable:** `Churn` (Yes/No)

---

## 2. Business Logic & Hypotheses

Before cleaning, we identify key variables that likely drive customer behavior:

1. **Contract Type:** Month-to-month customers usually have higher churn rates due to lower switching costs.
2. **Tenure:** New customers are higher risk; long-term customers demonstrate brand loyalty.
3. **Monthly Charges:** High costs relative to service value often trigger "price-sensitive" churn.
4. **Tech Support:** Customers without support services may leave due to unresolved technical frustrations.
5. **Internet Service:** Differences in technology (Fiber optic vs. DSL) may correlate with different satisfaction levels.

---

## 3. Data Integrity & Type Audit

A preliminary check of the data structure reveals two critical technical hurdles that must be addressed before any visualization or modeling:

### Critical Issue: `TotalCharges` Data Type

Logically, `TotalCharges` should be a numeric float. However, the initial `df.info()` shows it as an **object (string)**.

* **The Cause:** Likely contains empty strings `" "` or hidden non-numeric characters.
* **The Risk:** We cannot perform mathematical operations or correlations until this is cast to a numeric type.

### Redundant Feature: `customerID`

* **Observation:** Features like `7590-VHVEG` are unique identifiers.
* **Action:** This column provides no predictive power and will be dropped to prevent the model from "memorizing" specific IDs (overfitting).

---

## 4. Summary of Observations

| Feature | Type | Status | Note |
| --- | --- | --- | --- |
| **Churn** | Categorical | Target | Needs conversion to binary (0/1) |
| **TotalCharges** | Object | **Action Required** | Convert to Numeric; handle nulls |
| **customerID** | Object | **Action Required** | Drop during preprocessing |
| **Tenure** | Integer | Feature | Key indicator of customer lifecycle |

---


In [7]:
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [8]:
(df["TotalCharges"] == " ").sum()

np.int64(11)

### Detecting "Hidden" Missing Values

### 1. The `isnull()` Trap

Our initial check returned **0** missing values. However, we identified that `TotalCharges` is stored as an **Object (string)** rather than a **Float**. In real-world data, "missing" doesn't always mean `NaN`; it often appears as:

* Empty strings (`" "`)
* Placeholders like `"?"` or `"N/A"`

### 2. Investigation Result

By specifically querying for empty strings, we found the culprit:

```python
(df["TotalCharges"] == " ").sum()
# Output: 11

```

**Finding:** There are **11 rows** where `TotalCharges` is a blank space. Because these are strings, Pandas cannot perform calculations on this column, and a standard `isnull()` check will not detect them.

---

### 3. Root Cause Analysis

If we look at these 11 rows, we typically find that their **`tenure` is 0**.

* **Logic:** These are brand-new customers who have joined but have not yet been billed for their first month.
* **Problem:** We cannot convert the column to numeric until these strings are handled.

---


### The "New Customer" Hypothesis

### 1. Data Integrity Summary

| Issue | Status |
| --- | --- |
| **Total Rows** | 7,043 |
| **Blank `TotalCharges` Rows** | 11 |
| **Current Column Type** | `object` (Incorrect) |

### 2. Analytical Thinking: Why the Blanks?

In data science, we don't just "delete" errors; we seek their origin. The relationship between charges and time is typically defined by:

$$\text{TotalCharges} \approx \text{MonthlyCharges} \times \text{tenure}$$

**The Hypothesis:**
If a customer has a `tenure` of **0 months**, they are in their first billing cycle. In many legacy telecom databases, "zero accumulated charges" is stored as a **null string (" ")** rather than a numerical `0`.

